# Streaming Responses - Real-Time Output

## Purpose
Learn how to stream agent responses token-by-token for real-time user experiences. Streaming provides immediate feedback and makes applications feel more responsive, especially for longer responses.

## Key Concepts
- **Streaming**: Incremental token-by-token output delivery
- **Runner.run_streamed()**: Method that returns streaming result
- **ResponseTextDeltaEvent**: Individual token delta events
- **Async Iteration**: Process events as they arrive

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
model_id = "openai.gpt-5.5"

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Import Libraries

Import `ResponseTextDeltaEvent` to process token deltas:

In [ ]:
import asyncio
from agents import Agent, Runner
from openai.types.responses import ResponseTextDeltaEvent

## Step 1: Create an Agent

Create a regular agent - streaming works with any agent configuration:

In [ ]:
agent = Agent(
    name="Assistant",
    model=model_id,
    instructions="Always give 500 words response",
)

## Step 2: Stream the Response

Use `Runner.run_streamed()` instead of `Runner.run()`:

**How it works**:
1. `Runner.run_streamed()` returns immediately with a result object
2. Call `result.stream_events()` to get an async iterator
3. Iterate through events as they arrive
4. Check for `ResponseTextDeltaEvent` to get token deltas
5. Print tokens immediately for real-time display

💡 **Key Details**:
- `end=""` keeps tokens on same line
- `flush=True` displays immediately (no buffering)
- `event.data.delta` contains the token text

🔍 **Watch**: Tokens appear one by one as they're generated!

In [ ]:
result = Runner.run_streamed(agent, "Summarize recursion in one sentence.")

async for event in result.stream_events():
    # Token-level text deltas come through as raw_response_event
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

print()  # final newline

## 🎉 Congratulations!

You've completed the **Streaming Responses** notebook!